# Test Suite: Explainable AI (XAI) & Attribution Module
This notebook executes the unit tests and interactive validations for `src/explainability.py`.
It tests:
- Feature names extraction from multi-modal pipelines
- Global feature importance extraction (log-odds linear coefficients)
- Local single-review feature attribution (`explain_single_prediction`)


In [1]:
import os
import sys
import unittest
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline

# Ensure project root is in sys.path
project_root = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
if project_root not in sys.path:
    sys.path.insert(0, project_root)
os.chdir(project_root)

print(f"Working Directory: {os.getcwd()}")
from src.explainability import (
    get_feature_names_from_pipeline,
    extract_global_feature_importance,
    explain_single_prediction
)
print("Explainability module imported successfully!")


Working Directory: e:\ML\mini project 2
Explainability module imported successfully!


## Interactive Demonstration 1: Feature Names Extraction


In [2]:
texts = [
    "This genuine vacuum cleaner works wonderfully and reliably.",
    "Great quality product, authentic seller.",
    "Total fake scam, defective item, do not buy.",
    "Horrible fake garbage, broken immediately."
]
labels = [0, 0, 1, 1]

pipeline = Pipeline([
    ("features", TfidfVectorizer()),
    ("classifier", LogisticRegression())
])
pipeline.fit(texts, labels)

fnames = get_feature_names_from_pipeline(pipeline)
print(f"Extracted {len(fnames)} features: {fnames}")


Extracted 25 features: ['and', 'authentic', 'broken', 'buy', 'cleaner', 'defective', 'do', 'fake', 'garbage', 'genuine', 'great', 'horrible', 'immediately', 'item', 'not', 'product', 'quality', 'reliably', 'scam', 'seller', 'this', 'total', 'vacuum', 'wonderfully', 'works']


## Interactive Demonstration 2: Global Feature Importance (Coefficients)


In [3]:
top_fake, top_honest = extract_global_feature_importance(pipeline, top_n=3, save_csv=False)
print("Top Fake Predictors (+ log-odds):")
display(top_fake)
print("Top Honest Predictors (- log-odds):")
display(top_honest)


Top Fake Predictors (+ log-odds):


,feature,coefficient,odds_ratio
7,fake,0.258992,1.295623
8,garbage,0.184684,1.202838
2,broken,0.184684,1.202838


Top Honest Predictors (- log-odds):


,feature,coefficient,odds_ratio
1,authentic,-0.17756,0.837311
10,great,-0.17756,0.837311
15,product,-0.17756,0.837311


## Interactive Demonstration 3: Local Single-Review Attribution


In [4]:
review = "This fake product is a total scam!"
explanation = explain_single_prediction(pipeline, review, top_k=3)
print(f"Review: '{explanation['text']}'")
print(f"Predicted Class: {explanation['predicted_class']} ({explanation['predicted_label']})")
print(f"Probability Fake: {explanation['probability_fake']}")
print("\nTop Suspicious Cues:")
for cue in explanation['suspicious_cues']:
    print(f"  {cue['feature']:20s} (score: {cue['contribution']:+.4f})")


Review: 'This fake product is a total scam!'
Predicted Class: 1 (Likely Suspicious)
Probability Fake: 0.515

Top Suspicious Cues:
  fake                 (score: +0.0950)
  scam                 (score: +0.0669)
  total                (score: +0.0669)


## Formal Unittest Suite Runner


In [5]:
class TestExplainability(unittest.TestCase):

    def setUp(self):
        self.texts = [
            "This genuine vacuum cleaner works wonderfully and reliably.",
            "Great quality product, authentic seller.",
            "Total fake scam, defective item, do not buy.",
            "Horrible fake garbage, broken immediately."
        ]
        self.labels = [0, 0, 1, 1]

        self.pipeline = Pipeline([
            ("features", TfidfVectorizer()),
            ("classifier", LogisticRegression())
        ])
        self.pipeline.fit(self.texts, self.labels)

    def test_feature_names_extraction(self):
        fnames = get_feature_names_from_pipeline(self.pipeline)
        self.assertGreater(len(fnames), 5)
        self.assertIn("vacuum", fnames)
        self.assertIn("fake", fnames)

    def test_global_feature_importance(self):
        top_fake, top_honest = extract_global_feature_importance(self.pipeline, top_n=3, save_csv=False)
        self.assertEqual(len(top_fake), 3)
        self.assertEqual(len(top_honest), 3)
        self.assertIn("coefficient", top_fake.columns)
        self.assertIn("odds_ratio", top_fake.columns)

    def test_explain_single_prediction(self):
        review = "This fake product is a total scam!"
        explanation = explain_single_prediction(self.pipeline, review, top_k=3)
        self.assertEqual(explanation["predicted_class"], 1)
        self.assertEqual(explanation["predicted_label"], "Likely Suspicious")
        self.assertIsInstance(explanation["probability_fake"], float)
        self.assertGreater(len(explanation["suspicious_cues"]), 0)

# Run unittest suite in notebook
suite = unittest.TestLoader().loadTestsFromTestCase(TestExplainability)
runner = unittest.TextTestRunner(verbosity=2)
result = runner.run(suite)
assert result.wasSuccessful(), "Some tests failed!"


test_explain_single_prediction (__main__.TestExplainability.test_explain_single_prediction) ... ok
test_feature_names_extraction (__main__.TestExplainability.test_feature_names_extraction) ... ok
test_global_feature_importance (__main__.TestExplainability.test_global_feature_importance) ... ok

----------------------------------------------------------------------
Ran 3 tests in 0.031s

OK
